# От ноутбука к версии модели

Цель: прочитать клиентов из PostgreSQL, обучить простую модель оттока и сохранить **кандидата** в MLflow. Данные учебные и заполняются при запуске стенда; Airflow для этого шага не нужен. Запускайте ячейки сверху вниз.

## 1. Подключиться к MLflow

В браузере открывайте [MLflow на localhost:5000](http://localhost:5000). Имя `mlflow` работает только внутри сети Docker; если оно появится в автоматическом выводе MLflow, замените его на `localhost`.

In [1]:
import os

import mlflow
import mlflow.sklearn
import pandas as pd
import psycopg2
from mlflow.models import infer_signature
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
mlflow.set_experiment("notebook-churn-demo")
print("MLflow в браузере: http://localhost:5000")

MLflow в браузере: http://localhost:5000


## 2. Прочитать клиентов из PostgreSQL

Таблица `app.customers` содержит три признака клиента и исторический факт оттока `churn`. Запрос можно повторить в DBeaver. Для нового клиента метка оттока неизвестна: сервис получит только признаки.

In [2]:
query = """
SELECT tenure_months, monthly_spend, support_tickets, churn
FROM app.customers
ORDER BY customer_id
"""
with psycopg2.connect(os.environ["DEMO_DATABASE_URL"]) as connection:
    with connection.cursor() as cursor:
        cursor.execute(query)
        clients = pd.DataFrame(cursor.fetchall(), columns=[
            "tenure_months", "monthly_spend", "support_tickets", "churn"
        ])

print(f"Клиентов в PostgreSQL: {len(clients)}")
assert len(clients) >= 100 and clients.churn.nunique() == 2
clients.head()

Клиентов в PostgreSQL: 1200


,tenure_months,monthly_spend,support_tickets,churn
0,7,93.39,6,0
1,7,27.89,6,0
2,53,159.35,6,1
3,61,80.45,4,0
4,57,166.32,5,1


## 3. Обучить и проверить

Часть клиентов оставляем для проверки качества. ROC AUC показывает, насколько модель разделяет клиентов с оттоком и без него.

In [3]:
x_train, x_test, y_train, y_test = train_test_split(
    clients[["tenure_months", "monthly_spend", "support_tickets"]],
    clients["churn"], test_size=0.25, random_state=117, stratify=clients["churn"]
)
model = LogisticRegression(max_iter=500)
model.fit(x_train, y_train)
roc_auc = roc_auc_score(y_test, model.predict_proba(x_test)[:, 1])
print(f"ROC AUC: {roc_auc:.3f}")
assert roc_auc >= 0.65, "Качество ниже порога для демонстрации"

ROC AUC: 0.834


## 4. Сохранить кандидата

MLflow запишет параметры, метрику и модель. Сам файл модели попадёт в S3 через сервер MLflow. Регистрация создаст следующую версию `churn-prediction`, но **не переключит** alias `production`.

In [5]:
with mlflow.start_run(run_name="jupyter-logistic-regression") as run:
    mlflow.set_tags({
        "training_source": "notebook",
        "model_contract": "churn-v1",
        "dataset_version": "synthetic-v1",
        "dataset_source": "postgres.app.customers",
    })
    mlflow.log_params({"algorithm": "LogisticRegression", "train_rows": len(x_train)})
    mlflow.log_metric("roc_auc", float(roc_auc))
    logged = mlflow.sklearn.log_model(
        model, name="model_log", input_example=x_test.head(2),
        signature=infer_signature(x_test, model.predict(x_test)),
    )
    version = mlflow.register_model(logged.model_uri, "churn-prediction")
    client = mlflow.MlflowClient()
    client.set_model_version_tag("churn-prediction", version.version, "validation_status", "passed")
    client.set_model_version_tag("churn-prediction", version.version, "model_contract", "churn-v1")

print(f"Run: {run.info.run_id}")
print(f"Открыть run: http://localhost:5000/#/experiments/{run.info.experiment_id}/runs/{run.info.run_id}")
print(f"Новая версия churn-prediction: v{version.version}")
print("Alias production не менялся")

/usr/local/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/10/12 06:51:03 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during de

🏃 View run jupyter-logistic-regression at: http://mlflow:5000/#/experiments/3/runs/eee873844ebe48b5b35df4f6a79b15fa
🧪 View experiment at: http://mlflow:5000/#/experiments/3
Run: eee873844ebe48b5b35df4f6a79b15fa
Открыть run: http://localhost:5000/#/experiments/3/runs/eee873844ebe48b5b35df4f6a79b15fa
Новая версия churn-prediction: v19
Alias production не менялся


Created version '19' of model 'churn-prediction'.


## 5. Проверить результат

Откройте [MLflow → Models → churn-prediction](http://localhost:5000/#/models/churn-prediction). Найдите новую версию и её run в эксперименте `notebook-churn-demo`. В [S3-консоли](http://localhost:9001) откройте bucket `mlflow-artifacts` и файлы модели.

Дальше можно сравнить кандидата с текущей версией и вручную назначить ему alias `production` в MLflow. После этого откройте [ML-сервис `/model`](http://localhost:8000/model) и сделайте прогноз через [API](http://localhost:8000/docs). Повторный запуск ячеек создаст ещё один run и новую версию — это ожидаемо.